# LUNA16 3D Lung Nodule Detection

This notebook builds a **3D Convolutional Neural Network (CNN)** to classify lung nodules from volumetric CT scans sourced from the **LUNA16 dataset** (LUng Nodule Analysis 2016). The full pipeline includes:

1. Dataset setup and CT scan loading
2. Annotation-driven data exploration and nodule visualization
3. Positive (nodule) and negative (non-nodule) 3D patch extraction
4. Preprocessing, normalization, and 10-fold cross-validation splitting
5. PyTorch DataLoader construction
6. A 4-block 3D CNN model with BatchNorm and Dropout
7. Weighted cross-entropy training with learning rate scheduling
8. Evaluation via confusion matrix, classification report, and ROC-AUC
9. Model export


## 1. Setup

Define paths for the LUNA16 dataset mounted at `/kaggle/input/datasets/vafaeii/luna16`. Build a list of 10 subset folder paths (`subset0` through `subset9`), and set paths for `annotations.csv` (ground-truth nodule locations) and `candidates.csv`. Then count the total number of `.mhd` CT scan files across all subsets to confirm the dataset is accessible.


In [ ]:


import os

BASE_PATH = "/kaggle/input/datasets/vafaeii/luna16"

scan_folders = [
    os.path.join(
        BASE_PATH,
        f"subset{i}",
        f"subset{i}"
    )
    for i in range(10)
]

annotations_path = os.path.join(
    BASE_PATH,
    "annotations.csv"
)

candidates_path = os.path.join(
    BASE_PATH,
    "candidates.csv"
)



print("Scan folders loaded:")
for folder in scan_folders:
    print(folder)

scan_count = 0

for folder in scan_folders:
    scan_count += len([
        f for f in os.listdir(folder)
        if f.endswith(".mhd")
    ])

print("\nTotal CT scans:", scan_count)

### Imports & First Scan Load

Import core libraries: `SimpleITK` for reading `.mhd` volumetric CT files, `numpy` for array operations, `matplotlib` for visualization, `glob` for file discovery, and `os` for path handling. A confirmation message is printed to verify all imports succeed before proceeding.


In [ ]:
import SimpleITK as sitk
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

print("Everything imported successfully!")



## 2. Data Exploration

### Load Annotations

Read `annotations.csv` into a pandas DataFrame. Each row represents a confirmed nodule and contains:
- `seriesuid` — unique identifier matching a `.mhd` scan file
- `coordX`, `coordY`, `coordZ` — nodule center in world (mm) coordinates
- `diameter_mm` — nodule size in millimetres

Print the total nodule count and preview the first few rows.


In [ ]:
import pandas as pd
import os



BASE_PATH = "/kaggle/input/datasets/vafaeii/luna16"

annotations_path = os.path.join(
    BASE_PATH,
    "annotations.csv"
)

annotations = pd.read_csv(
    annotations_path
)

print("Total nodules:", len(annotations))

annotations.head()

### Find Nodules for a Single Scan

List all `.mhd` scan files in `subset0` using `glob`. Extract the `seriesuid` from the first scan's filename (by stripping the `.mhd` extension) and filter the annotations DataFrame to retrieve all nodules belonging to that scan.


In [ ]:
import os
import glob
import SimpleITK as sitk


scan_path = (
    "/kaggle/input/datasets/"
    "vafaeii/luna16/subset0/subset0"
)


scan_files = glob.glob(
    os.path.join(scan_path, "*.mhd")
)

print("Total scans:", len(scan_files))
print("First scan:", scan_files[0])

In [ ]:
import os

# Get scan filename without extension
scan_id = os.path.basename(scan_files[0]).replace(".mhd", "")

print("Current Scan ID:")
print(scan_id)

# Find annotations for this scan
scan_nodules = annotations[
    annotations["seriesuid"] == scan_id
]

print("Nodules found:")
print(scan_nodules)

### Visualize a Nodule on the CT Slice

Load the first scan using `SimpleITK` and convert it to a NumPy array (shape: `[Z, Y, X]`). Extract the scan's `origin` and `spacing` metadata, then convert the nodule's world coordinate to a voxel index:

```
voxel_coord = (world_coord - origin) / spacing
```

Display the axial slice at the nodule's Z index and overlay a red circle whose radius equals `diameter_mm / spacing[0]` to mark the nodule location.


In [ ]:
import SimpleITK as sitk
import os


scan_file = scan_files[0]

scan = sitk.ReadImage(
    scan_file
)


scan_array = sitk.GetArrayFromImage(
    scan
)

print("Loaded scan!")
print("Shape:", scan_array.shape)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

origin = np.array(scan.GetOrigin())
spacing = np.array(scan.GetSpacing())

print("Origin:", origin)
print("Spacing:", spacing)


nodule = scan_nodules.iloc[0]

world_coord = np.array([
    nodule["coordX"],
    nodule["coordY"],
    nodule["coordZ"]
])

diameter = nodule["diameter_mm"]

print("World coordinate:", world_coord)
print("Diameter (mm):", diameter)


voxel_coord = (
    world_coord - origin
) / spacing

print("Voxel coordinate:", voxel_coord)


slice_idx = int(voxel_coord[2])


plt.figure(figsize=(8,8))
plt.imshow(
    scan_array[slice_idx],
    cmap="gray"
)


circle = plt.Circle(
    (
        voxel_coord[0],
        voxel_coord[1]
    ),
    diameter / spacing[0],
    color="red",
    fill=False,
    linewidth=2
)

plt.gca().add_patch(circle)

plt.title(
    f"Nodule Slice: {slice_idx}"
)

plt.axis("off")
plt.show()

### Extract 3D Nodule Cube (Positive Sample)

Crop a **64×64×64 voxel cube** (±32 voxels in each axis) centered on the nodule's voxel coordinate. Boundary clamping with `max(0, ...)` and `min(shape, ...)` prevents out-of-bounds indexing. Display the middle axial slice of the extracted cube to visually confirm the crop is centered on the nodule.


In [ ]:

x, y, z = voxel_coord.astype(int)


cube_size = 32


z_min = max(0, z - cube_size)
z_max = min(scan_array.shape[0], z + cube_size)

y_min = max(0, y - cube_size)
y_max = min(scan_array.shape[1], y + cube_size)

x_min = max(0, x - cube_size)
x_max = min(scan_array.shape[2], x + cube_size)


nodule_cube = scan_array[
    z_min:z_max,
    y_min:y_max,
    x_min:x_max
]

print("Cube shape:", nodule_cube.shape)

middle = nodule_cube.shape[0] // 2

plt.figure(figsize=(6,6))
plt.imshow(
    nodule_cube[middle],
    cmap="gray"
)

plt.title("Extracted Nodule Region")
plt.axis("off")
plt.show()

## 3. Negative Sample Strategy

### Naive Negative Sampling (Initial Approach)

Pick a random `(z, y, x)` location within the scan volume (with a 50-voxel margin from the edges) and extract a 64×64×64 cube. This baseline approach is simple but has two problems: the sampled region may overlap with an annotated nodule, or it may land outside the lung entirely (e.g., in background air or the chest wall).


In [ ]:
import random


z_rand = random.randint(50, scan_array.shape[0]-50)
y_rand = random.randint(50, scan_array.shape[1]-50)
x_rand = random.randint(50, scan_array.shape[2]-50)

cube_size = 32

negative_cube = scan_array[
    z_rand-cube_size:z_rand+cube_size,
    y_rand-cube_size:y_rand+cube_size,
    x_rand-cube_size:x_rand+cube_size
]

print("Negative cube shape:", negative_cube.shape)

middle = negative_cube.shape[0] // 2

plt.figure(figsize=(6,6))
plt.imshow(negative_cube[middle], cmap="gray")
plt.title("Negative Sample")
plt.axis("off")
plt.show()

### Improved Negative Sampling

Iterate up to **1000 attempts** to find a valid negative sample that satisfies two constraints:

1. **Spatial distance** — the candidate center must be at least **80 voxels away** from every known nodule voxel coordinate (Euclidean distance).
2. **Tissue check** — the mean HU value of the extracted 64×64×64 cube must be **below −300 HU**, confirming the patch falls inside aerated lung tissue rather than the chest wall, mediastinum, or background.

If both conditions are met and the cube is exactly 64×64×64, the loop breaks and returns the valid negative patch.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

cube_size = 32


x_true, y_true, z_true = voxel_coord.astype(int)

max_attempts = 1000
attempt = 0

while attempt < max_attempts:

    z_rand = random.randint(
        cube_size,
        scan_array.shape[0] - cube_size
    )

    y_rand = random.randint(
        cube_size,
        scan_array.shape[1] - cube_size
    )

    x_rand = random.randint(
        cube_size,
        scan_array.shape[2] - cube_size
    )

  
    dist = np.sqrt(
        (x_rand - x_true)**2 +
        (y_rand - y_true)**2 +
        (z_rand - z_true)**2
    )

    if dist > 80:

        negative_cube = scan_array[
            z_rand-cube_size:z_rand+cube_size,
            y_rand-cube_size:y_rand+cube_size,
            x_rand-cube_size:x_rand+cube_size
        ]

     
        if negative_cube.shape == (
            cube_size*2,
            cube_size*2,
            cube_size*2
        ):

            if np.mean(negative_cube) < -300:
                break

    attempt += 1

print("Negative cube shape:",
      negative_cube.shape)

middle = negative_cube.shape[0] // 2

plt.figure(figsize=(6,6))
plt.imshow(
    negative_cube[middle],
    cmap="gray"
)

plt.title(
    "Better Negative Lung Sample"
)

plt.axis("off")
plt.show()

## 4. Helper Functions

Define three reusable utility functions used throughout dataset construction:

- **`world_to_voxel(world_coord, origin, spacing)`** — converts a 3D world coordinate (mm) to a voxel index by computing `(world_coord - origin) / spacing`, cast to `int`.
- **`extract_cube(scan_array, center, cube_size=32)`** — crops a `(2×cube_size)³` = **64³** voxel cube around a given center, with boundary clamping to avoid index errors.
- **`sample_negative(scan_array, positive_centers)`** — repeatedly samples random scan locations (up to 1000 attempts), accepting only those that are >80 voxels from all nodule centers AND have a mean HU < −300 AND produce a full 64×64×64 cube.


In [ ]:
import numpy as np
import random

CUBE_SIZE = 32


def world_to_voxel(
    world_coord,
    origin,
    spacing
):
    voxel_coord = (
        world_coord - origin
    ) / spacing

    return voxel_coord.astype(int)


def extract_cube(
    scan_array,
    center,
    cube_size=CUBE_SIZE
):
    x, y, z = center

    z1 = max(0, z - cube_size)
    z2 = min(
        scan_array.shape[0],
        z + cube_size
    )

    y1 = max(0, y - cube_size)
    y2 = min(
        scan_array.shape[1],
        y + cube_size
    )

    x1 = max(0, x - cube_size)
    x2 = min(
        scan_array.shape[2],
        x + cube_size
    )

    cube = scan_array[
        z1:z2,
        y1:y2,
        x1:x2
    ]

    return cube


def sample_negative(
    scan_array,
    positive_centers
):

    max_attempts = 1000
    attempt = 0

    while attempt < max_attempts:

        z = random.randint(
            CUBE_SIZE,
            scan_array.shape[0] - CUBE_SIZE
        )

        y = random.randint(
            CUBE_SIZE,
            scan_array.shape[1] - CUBE_SIZE
        )

        x = random.randint(
            CUBE_SIZE,
            scan_array.shape[2] - CUBE_SIZE
        )

        safe = True

        for center in positive_centers:

            px, py, pz = center

            dist = np.sqrt(
                (x - px)**2 +
                (y - py)**2 +
                (z - pz)**2
            )

            if dist < 80:
                safe = False
                break

        if not safe:
            attempt += 1
            continue

        cube = extract_cube(
            scan_array,
            (x, y, z)
        )

        if cube.shape == (
            64,
            64,
            64
        ):

            if np.mean(cube) < -300:
                return cube

        attempt += 1

    return None


print(
    "Helper functions loaded successfully!"
)

## 5. Dataset Construction — All Subsets

Iterate over **all 10 subset folders** (`subset0` through `subset9`). For each `.mhd` scan file:

- Load the scan with `SimpleITK`, extract the array, origin, and spacing.
- Look up all annotated nodules for that `seriesuid` in the annotations DataFrame.
- For each nodule: convert world→voxel, extract a 64³ cube → **label 1** (positive).
- For each scan: sample `max(2, num_nodules × 2)` negative cubes using `sample_negative` → **label 0**.
- All cubes are cast to `float16` before appending to keep memory usage manageable.

At the end, print the total sample count and the positive/negative breakdown.


In [ ]:
import glob
import os
import SimpleITK as sitk
import numpy as np

X = []
y = []

BASE_PATH = (
    "/kaggle/input/datasets/"
    "vafaeii/luna16"
)

scan_folders = [
    os.path.join(
        BASE_PATH,
        f"subset{i}",
        f"subset{i}"
    )
    for i in range(10)
]

for folder in scan_folders:

    print("\nProcessing:", folder)

    scan_files = glob.glob(
        os.path.join(folder, "*.mhd")
    )

    for idx, scan_file in enumerate(scan_files):

        if idx % 10 == 0:
            print(f"Scan {idx}")

        scan_id = os.path.basename(
            scan_file
        ).replace(".mhd", "")

        scan = sitk.ReadImage(
            scan_file
        )

        scan_array = sitk.GetArrayFromImage(
            scan
        )

        origin = np.array(
            scan.GetOrigin()
        )

        spacing = np.array(
            scan.GetSpacing()
        )

        nodules = annotations[
            annotations["seriesuid"]
            == scan_id
        ]

        positive_centers = []

       
        for _, row in nodules.iterrows():

            world_coord = np.array([
                row["coordX"],
                row["coordY"],
                row["coordZ"]
            ])

            voxel_coord = world_to_voxel(
                world_coord,
                origin,
                spacing
            )

            positive_centers.append(
                voxel_coord
            )

            cube = extract_cube(
                scan_array,
                voxel_coord
            )

            if cube.shape == (
                64,
                64,
                64
            ):

                cube = cube.astype(
                    np.float16
                )

                X.append(cube)
                y.append(1)

       
        for _ in range(
            max(
                2,
                len(positive_centers) * 2
            )
        ):

            neg_cube = sample_negative(
                scan_array,
                positive_centers
            )

            if neg_cube is None:
                continue

            neg_cube = neg_cube.astype(
                np.float16
            )

            X.append(neg_cube)
            y.append(0)

print("\nDONE")
print("Total samples:", len(X))
print("Positive:", sum(y))
print(
    "Negative:",
    len(y) - sum(y)
)

## 6. Preprocessing

Convert the accumulated lists to NumPy arrays and apply the following steps:

1. **HU clipping** — clip values to `[−1000, +400]` HU (standard lung window).
2. **Normalization** — shift and scale to `[0, 1]` via `(X + 1000) / 1400`.
3. **Cast to float32** — required for PyTorch tensor operations.
4. **Channel dimension** — add axis 1 → shape becomes `(N, 1, 64, 64, 64)`, as expected by `Conv3d`.
5. **10-Fold Stratified Cross-Validation** — use `StratifiedKFold(n_splits=10, shuffle=True, random_state=42)` to split data while preserving the class ratio in every fold. Print the train/test shapes for each fold.


In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold


X = np.array(
    X,
    dtype=np.float16
)

y = np.array(y)

print(
    "Raw dataset shape:",
    X.shape
)

print(
    "Labels shape:",
    y.shape
)



X = np.clip(
    X,
    -1000,
    400
)

X = (
    X + 1000
) / 1400

X = X.astype(
    np.float32
)

# Add channel dimension
X = np.expand_dims(
    X,
    axis=1
)

print(
    "Normalized shape:",
    X.shape
)


kfold = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

print("\n10-Fold CV Ready!")

for fold, (
    train_idx,
    test_idx
) in enumerate(
    kfold.split(X, y)
):

    X_train = X[
        train_idx
    ]

    X_test = X[
        test_idx
    ]

    y_train = y[
        train_idx
    ]

    y_test = y[
        test_idx
    ]

    print(
        f"\nFold {fold+1}"
    )

    print(
        "Train:",
        X_train.shape
    )

    print(
        "Test:",
        X_test.shape
    )

## 7. PyTorch DataLoaders

For each of the 10 folds, convert the NumPy splits into PyTorch tensors (`dtype=torch.float32` for inputs, `dtype=torch.long` for labels) and wrap them in `TensorDataset` objects. Then create:

- **`train_loader`** — `DataLoader` with `batch_size=8` and `shuffle=True`
- **`test_loader`** — `DataLoader` with `batch_size=4` and `shuffle=False`

Using `torch.float32` is required because PyTorch's `Conv3d` does not support `float16` inputs on CPU. Batch size 8 is chosen to fit GPU VRAM while maintaining stable gradient estimates.


In [ ]:
import torch
from torch.utils.data import (
    TensorDataset,
    DataLoader
)



for fold, (
    train_idx,
    test_idx
) in enumerate(
    kfold.split(X, y)
):

    print(
        f"\n========== Fold {fold+1} =========="
    )

    # split data
    X_train = X[
        train_idx
    ]

    X_test = X[
        test_idx
    ]

    y_train = y[
        train_idx
    ]

    y_test = y[
        test_idx
    ]

  

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.long
    )

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32
    )

    y_test_tensor = torch.tensor(
        y_test,
        dtype=torch.long
    )

   
    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )

    test_dataset = TensorDataset(
        X_test_tensor,
        y_test_tensor
    )

  
    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=4,
        shuffle=False
    )

    print(
        "PyTorch data ready!"
    )

    print(
        "Train shape:",
        X_train_tensor.shape
    )

    print(
        "Test shape:",
        X_test_tensor.shape
    )

## 8. Model Architecture — 3D CNN

`LungNoduleCNN` is a **4-block 3D CNN** defined in PyTorch:

| Block | Layer | Output Channels | Kernel | Pool |
|-------|-------|----------------|--------|------|
| 1 | Conv3d + BN + ReLU + MaxPool | 16 | 3×3×3 | 2×2×2 |
| 2 | Conv3d + BN + ReLU + MaxPool | 32 | 3×3×3 | 2×2×2 |
| 3 | Conv3d + BN + ReLU + MaxPool | 64 | 3×3×3 | 2×2×2 |
| 4 | Conv3d + BN + ReLU + MaxPool | 128 | 3×3×3 | 2×2×2 |

After 4× halving, a 64³ input becomes **4×4×4** feature maps → flattened to `128 × 4 × 4 × 4 = 8192` features.

The fully connected head is: `Linear(8192, 256) → ReLU → Dropout(0.4) → Linear(256, 2)`.

**BatchNorm3d** accelerates convergence and reduces sensitivity to initialization. **Dropout(0.4)** mitigates overfitting on a relatively small medical dataset. The model is moved to GPU if CUDA is available.


In [ ]:
import torch
import torch.nn as nn


class LungNoduleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv3d(
                1, 16, 3,
                padding=1
            ),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                16, 32, 3,
                padding=1
            ),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                32, 64, 3,
                padding=1
            ),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                64, 128, 3,
                padding=1
            ),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.fc = nn.Sequential(

            nn.Linear(
                128 * 4 * 4 * 4,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(
                256,
                2
            )
        )

    def forward(self, x):

        x = self.conv(x)

        x = x.view(
            x.size(0),
            -1
        )

        x = self.fc(x)

        return x


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

## 9. Loss Function & Optimizer

Configure the training components for each fold:

- **Model** — fresh `LungNoduleCNN()` instance per fold to prevent weight leakage.
- **Loss** — `CrossEntropyLoss` with class weights `[1.0, 6.0]`. The 6× weight on the positive (nodule) class compensates for class imbalance and makes the model penalise missed nodules (false negatives) more heavily — the safer trade-off in a clinical screening context.
- **Optimizer** — `Adam` with `lr=1e-4` and `weight_decay=1e-5` (L2 regularisation).
- **Scheduler** — `StepLR` with `step_size=4` and `gamma=0.5`, halving the learning rate every 4 epochs to fine-tune convergence in later training stages.


In [ ]:
import torch.optim as optim



model = LungNoduleCNN().to(
    device
)



class_weights = torch.tensor(
    [1.0, 6.0]
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)



optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

scheduler = (
    torch.optim.lr_scheduler
    .StepLR(
        optimizer,
        step_size=4,
        gamma=0.5
    )
)

print(
    "Training setup ready!"
)

## 10. Training — 10-Fold Cross-Validation

Run the full training loop across all 10 folds, each for **18 epochs**:

- Per epoch: iterate over `train_loader`, compute weighted cross-entropy loss, backpropagate, step the optimizer and scheduler.
- Track and print **running loss**, **train accuracy**, and **current learning rate** each epoch.
- After training each fold, evaluate on `test_loader` in `model.eval()` mode and record fold test accuracy.

After all 10 folds, print the **mean 10-fold test accuracy** using `np.mean(fold_metrics)`.


In [ ]:
fold_metrics = []

epochs = 18

for fold, (
    train_idx,
    test_idx
) in enumerate(
    kfold.split(X, y)
):

    print(
        f"\n========== Fold {fold+1}/10 =========="
    )

  

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

   

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.long
    )

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32
    )

    y_test_tensor = torch.tensor(
        y_test,
        dtype=torch.long
    )

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )

    test_dataset = TensorDataset(
        X_test_tensor,
        y_test_tensor
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=4,
        shuffle=False
    )

  
    model = LungNoduleCNN().to(
        device
    )

    class_weights = torch.tensor(
        [1.0, 6.0]
    ).to(device)

    criterion = (
        nn.CrossEntropyLoss(
            weight=class_weights
        )
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-5
    )

    scheduler = (
        torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=4,
            gamma=0.5
        )
    )

    train_accs = []
    train_losses = []

 

    for epoch in range(epochs):

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(
                device
            )

            labels = labels.to(
                device
            )

            optimizer.zero_grad()

            outputs = model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            running_loss += (
                loss.item()
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

        scheduler.step()

        train_acc = (
            100 *
            correct /
            total
        )

        train_accs.append(
            train_acc
        )

        train_losses.append(
            running_loss
        )

        current_lr = (
            optimizer.param_groups[0]['lr']
        )

        print(
            f"Fold {fold+1} | "
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {running_loss:.2f} "
            f"Train Acc: {train_acc:.2f}% "
            f"LR: {current_lr:.6f}"
        )

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                device
            )

            labels = labels.to(
                device
            )

            outputs = model(
                images
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    test_acc = (
        100 *
        correct /
        total
    )

    print(
        f"\nFold {fold+1} "
        f"Test Accuracy: "
        f"{test_acc:.2f}%"
    )

    fold_metrics.append(
        test_acc
    )

print(
    "\nAverage 10-Fold Accuracy:",
    np.mean(fold_metrics)
)

## 11. Evaluation

Re-run inference on the last fold's `test_loader` using `model.eval()` and `torch.no_grad()`. Instead of taking the argmax directly, apply `torch.softmax` and use a **lowered threshold of 0.35** on the nodule probability (`probs[:, 1]`) to increase sensitivity — flagging more true positives at the cost of slightly more false positives, which is preferable in a screening setting. Collect all predicted labels and ground-truth labels for downstream metric computation.


In [ ]:
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )

        outputs = model(
            images
        )

      
        probs = torch.softmax(
            outputs,
            dim=1
        )

        predicted = (
            probs[:,1] > 0.35
        ).long()

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

        all_preds.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

test_accuracy = (
    100 *
    correct /
    total
)

print(
    f"Fold {fold+1} "
    f"Test Accuracy: "
    f"{test_accuracy:.2f}%"
)

### Confusion Matrix

Compute and visualize the confusion matrix using `sklearn.metrics.confusion_matrix` and `seaborn.heatmap`. The matrix shows counts for:
- **TN** (top-left): correct non-nodule predictions
- **FP** (top-right): non-nodules wrongly predicted as nodules
- **FN** (bottom-left): missed nodules — the most clinically dangerous error
- **TP** (bottom-right): correctly detected nodules

The plot title includes `FN` and `FP` counts for quick reference.


In [ ]:
from sklearn.metrics import (
    confusion_matrix
)

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    all_labels,
    all_preds
)

tn, fp, fn, tp = cm.ravel()

plt.figure(
    figsize=(6,5)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Negative",
        "Positive"
    ],
    yticklabels=[
        "Negative",
        "Positive"
    ]
)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Actual"
)

plt.title(
    f"Fold {fold+1} "
    f"Confusion Matrix\n"
    f"FN={fn}, FP={fp}"
)

plt.show()

print(
    f"TN: {tn}"
)

print(
    f"FP: {fp}"
)

print(
    f"FN: {fn}"
)

print(
    f"TP: {tp}"
)

### Classification Report & ROC-AUC

Print a full `classification_report` showing **precision**, **recall**, and **F1-score** for both classes (Negative / Positive) with 4 decimal places. **Recall on class 1** (nodule sensitivity) is the primary metric for a medical screening task.

Then re-run inference to collect **softmax probabilities** for the positive class and compute:
- **`roc_auc_score`** — area under the ROC curve; guards against the degenerate single-class edge case.
- **ROC Curve plot** — FPR vs. TPR with AUC annotated in the legend, saved to `/kaggle/working/roc_curve_fold_{fold+1}.png` at 300 DPI.


In [ ]:
from sklearn.metrics import (
    classification_report
)

print(
    f"\nFold {fold+1} "
    f"Classification Report"
)

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=[
            "Negative",
            "Positive"
        ],
        digits=4
    )
)

In [ ]:
from sklearn.metrics import (
    roc_auc_score
)

import torch.nn.functional as F

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device
        )

        outputs = model(
            images
        )

        probs = F.softmax(
            outputs,
            dim=1
        )[:, 1]

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )



if len(
    np.unique(all_labels)
) > 1:

    auc = roc_auc_score(
        all_labels,
        all_probs
    )

    print(
        f"Fold {fold+1} "
        f"ROC-AUC: "
        f"{auc:.4f}"
    )

else:

    print(
        "ROC-AUC undefined "
        "(single class only)"
    )

In [ ]:
from sklearn.metrics import (
    roc_curve,
    auc
)

import matplotlib.pyplot as plt



if len(
    np.unique(all_labels)
) > 1:

    fpr, tpr, thresholds = roc_curve(
        all_labels,
        all_probs
    )

    roc_auc = auc(
        fpr,
        tpr
    )

    plt.figure(
        figsize=(6,6)
    )

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=(
            f"AUC = "
            f"{roc_auc:.4f}"
        )
    )

    plt.plot(
        [0,1],
        [0,1],
        linestyle="--"
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        f"Fold {fold+1} "
        f"ROC Curve"
    )

    plt.legend()

    plt.grid(True)

    plt.savefig(
        f"/kaggle/working/"
        f"roc_curve_fold_"
        f"{fold+1}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

else:

    print(
        "ROC curve undefined "
        "(single class only)"
    )

## 12. Save Model

Persist the trained model's learned parameters to disk with `torch.save(model.state_dict(), ...)`. Saving only the `state_dict` (a dict of tensor weights) rather than the full model object is the recommended PyTorch practice — it is portable across Python versions and decoupled from the class definition. The file is written to `/kaggle/working/lung_nodule_model_final.pth`.


In [ ]:
torch.save(
    model.state_dict(),
    "/kaggle/working/lung_nodule_model_final.pth"
)

print("Model saved!")